In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("netflix_titles.csv")
print("Shape:", df.shape)
df.head(3)

Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [3]:
df.columns.to_list()

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [5]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [6]:
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [7]:
df['show_id'].duplicated().sum()

np.int64(0)

In [9]:
df['type'].unique()

<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

In [10]:
df['rating'].unique()

<StringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',   '74 min',
   '84 min',   '66 min',       'NR',        nan, 'TV-Y7-FV',       'UR']
Length: 18, dtype: str

In [11]:
print(df['rating'].value_counts())

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


In [12]:
rating_errors = df['rating'].isin(['74 min', '84 min', '66 min'])

In [13]:
df.loc[rating_errors, 'duration'] = df.loc[rating_errors, 'rating']

# rows where rating_errors was true was placed in correct column from rating to duration

In [14]:
df.loc[rating_errors, 'rating'] = np.nan

In [15]:
df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")

In [16]:
df["rating"] = df["rating"].fillna("Unknown")
df["rating"] = df["rating"].str.strip()

In [17]:
df.isnull().sum()

show_id           0
type              0
title             0
director          0
cast              0
country         831
date_added       10
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

In [18]:
df['type'] = df['type'].str.strip().str.lower()
df['type'] = df['type'].replace({'tv show':'tv_show'})

##### Cleaning and grouping the rating column

In [19]:
df['rating'] = (df['rating'].str.strip().str.upper().str.replace(' ', '-', regex=False))

In [20]:
rating_map = {
    'TV-Y'     : 'Kids',
    'TV-Y7'    : 'Kids',
    'TV-Y7-FV' : 'Kids',
    'G'        : 'Kids',
    'TV-G'     : 'General',
    'PG'       : 'General',
    'TV-PG'    : 'General',
    'PG-13'    : 'Teen',
    'TV-14'    : 'Teen',
    'R'        : 'Adult',
    'TV-MA'    : 'Adult',
    'NC-17'    : 'Adult',
    'NR'       : 'Unrated',
    'UR'       : 'Unrated',
    'Unknown'  : 'Unknown'
}

df['rating_group'] = df['rating'].map(rating_map).fillna('Unknown')

In [21]:
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(),errors="coerce")

In [23]:
df['year_added']  = df['date_added'].dt.year.astype('Int64')

In [24]:
df['month_added'] = df['date_added'].dt.month.astype('Int64')

##### Creating primary and secondary country and genre

In [25]:
country_split = df['country'].str.split(',')

df['primary_country'] = country_split.str[0].str.strip()
df['secondary_country'] = country_split.str[1].str.strip()

In [26]:
df['primary_country'] = df['primary_country'].fillna('Unknown')
df['secondary_country'] = df['secondary_country'].fillna('None')

In [27]:
genre_split = df['listed_in'].str.split(',')

df['primary_genre'] = genre_split.str[0].str.strip()
df['sub_genre'] = genre_split.str[1].str.strip()

In [28]:
df['primary_genre'] = df['primary_genre'].fillna('Unknown')
df['sub_genre'] = df['sub_genre'].fillna('None')

In [31]:
duration_split = df['duration'].str.extract(r'(\d+)\s*(\w+)')

df['duration_int'] = duration_split[0].astype('float')
df['duration_type'] = duration_split[1]

In [32]:
df['movie_minutes'] = np.where(df['duration_type']=='min', df['duration_int'], np.nan)

df['tv_seasons'] = np.where(
    df['duration_type'].str.contains('Season', na=False),
    df['duration_int'],
    np.nan
)

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   show_id            8807 non-null   str           
 1   type               8807 non-null   str           
 2   title              8807 non-null   str           
 3   director           8807 non-null   str           
 4   cast               8807 non-null   str           
 5   country            7976 non-null   str           
 6   date_added         8797 non-null   datetime64[us]
 7   release_year       8807 non-null   int64         
 8   rating             8807 non-null   str           
 9   duration           8807 non-null   str           
 10  listed_in          8807 non-null   str           
 11  description        8807 non-null   str           
 12  rating_group       8807 non-null   str           
 13  year_added         8797 non-null   Int64         
 14  month_added        

In [35]:
df = df.dropna(subset=['date_added'])

In [36]:
df.isnull().sum()

show_id                 0
type                    0
title                   0
director                0
cast                    0
country               830
date_added              0
release_year            0
rating                  0
duration                0
listed_in               0
description             0
rating_group            0
year_added              0
month_added             0
primary_country         0
secondary_country       0
primary_genre           0
sub_genre               0
duration_int            0
duration_type           0
movie_minutes        2666
tv_seasons           6131
dtype: int64

##### column selection for final clean csv

In [37]:
final_columns = ['show_id','title','type','director','cast',
    'release_year','date_added','year_added','month_added',
    'rating','rating_group',
    'primary_genre','sub_genre',
    'primary_country','secondary_country',
    'duration_int','duration_type','movie_minutes','tv_seasons','description'
]


In [38]:
df_final = df[final_columns].copy()

df_final.to_csv('netflix_cleaned.csv', index=False)